In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader

import torchvision.transforms as transforms
import torchvision.datasets as datasets

from sklearn import metrics
from sklearn import decomposition
from sklearn import manifold
from tqdm.notebook import trange, tqdm
import matplotlib.pyplot as plt
import numpy as np

import copy
import random
import time


import os
from PIL import Image
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [ ]:
class CustomData(data.Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        """
        Args:
            csv_file (string): Path to the CSV file with annotations.
            image_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.annotations = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.annotations.iloc[idx, 0])
        image = Image.open(img_path).convert("RGB")
        label = self.annotations.iloc[idx, 1]

        if self.transform:
            image = self.transform(image)

        return image, label


In [ ]:
train_csv = '/content/drive/MyDrive/tensorflow/train/_annotations.csv'
train_image_dir = '/content/drive/MyDrive/tensorflow/train'

test_csv = '/content/drive/MyDrive/tensorflow/test/_annotations.csv'
test_image_dir = '/content/drive/MyDrive/tensorflow/test'

valid_csv = '/content/drive/MyDrive/tensorflow/valid/_annotations.csv'
valid_image_dir = '/content/drive/MyDrive/tensorflow/valid'


In [ ]:
# Define your transformations
train_transforms = transforms.Compose([
    transforms.RandomRotation(5, fill=(0,)),
    transforms.RandomCrop(28, padding=2),
    transforms.ToTensor()
])


In [ ]:
# Create an instance of CustomData for train_data
train_data = CustomData(csv_file=train_csv, image_dir=train_image_dir, transform=train_transforms)


In [ ]:
# Calculate mean and std for normalization
mean = 0.0
std = 0.0
total_images = len(train_data)

for images, _ in train_data:
    mean += torch.mean(images)
    std += torch.std(images)

mean /= total_images
std /= total_images

print("Mean:", mean)
print("Standard Deviation:", std)

Mean: tensor(0.4351)
Standard Deviation: tensor(0.0750)


In [ ]:
train_transforms = transforms.Compose([
                            transforms.RandomRotation(5, fill=(0,)),
                            transforms.RandomCrop(28, padding=2),
                            transforms.ToTensor(),
                            transforms.Normalize(mean=[mean], std=[std])
                                      ])

test_transforms = transforms.Compose([
                           transforms.ToTensor(),
                           transforms.Normalize(mean=[mean], std=[std])
                                     ])

In [ ]:
# Create instances of CustomData for your training, testing, and validation data
train_data = CustomData(csv_file=train_csv, image_dir=train_image_dir, transform=train_transforms)
test_data = CustomData(csv_file=test_csv, image_dir=test_image_dir, transform=test_transforms)
valid_data = CustomData(csv_file=valid_csv, image_dir=valid_image_dir, transform=test_transforms)


In [ ]:
# Define batch size
batch_size = 64

# Create data loaders
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
valid_loader = DataLoader(valid_data, batch_size=batch_size, shuffle=False)

In [ ]:
print(f'Number of training examples: {len(train_data)}')
print(f'Number of testing examples: {len(test_data)}')

Number of training examples: 6528
Number of testing examples: 124


In [ ]:
import torch.nn as nn
#define cnn model
class CNNModel(nn.Module):
    def __init__(self, num_classes=10):
        super(CNNModel, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)

        # Max pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        # Fully connected layers
        self.fc1 = nn.Linear(128 * 3 * 3, 512)
        self.fc2 = nn.Linear(512, num_classes)

        # Dropout layer
        self.dropout = nn.Dropout(0.25)

        # ReLU activation function
        self.relu = nn.ReLU()

    def forward(self, x):
        # Convolutional layers with max pooling and ReLU activation
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))

        # Flatten the output
        x = x.view(-1, 128 * 3 * 3)

        # Fully connected layers with dropout and ReLU activation
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x
